In [0]:
%run ../../config/utils

In [0]:
import sys
sys.path.append("..")
sys.path.append("../..")

from lib.job_manager import load_config, split_config
from lib_etl.s3 import etl_input_data_validator
import lib_etl.validations_ETL as validations
import pyspark.sql.functions as f
from pyspark.sql.types import *

In [0]:
config = load_config(etl_config_path)
data_paths, club_square_config, config_validation = split_config(config)

## Load source data

In [0]:
coupon_clip_schema = StructType(
    [
        StructField("usercode", StringType(), True),
        StructField("EVENTDATETIME", DateType(), True),
        StructField("EVENTTYPE", StringType(), True),
        StructField("OFFERID", StringType(), True),
        StructField("STORENAME", StringType(), True),
        StructField("OFFERCODE", StringType(), True),
        StructField("OFFERACTIVEDATE", DateType(), True),
        StructField("OFFERSHUTOFFDATE", DateType(), True),
        StructField("OFFEREXPIRYDATE", DateType(), True),
        StructField("BRAND", StringType(), True),
        StructField("OFFERVALUE", DoubleType(), True),
        StructField("OFFERTYPE", StringType(), True),
        StructField("DISCOUNTTYPE", StringType(), True),
        StructField("CATEGORY", StringType(), True),
        StructField("SOURCE", StringType(), True),
        StructField("APPID", StringType(), True),
        StructField("APPCODE", StringType(), True),
    ]
)
coupon_clip_path = data_paths["source"]["coupon_clip"]

In [0]:
df = (
        spark.read.schema(coupon_clip_schema)
        .option("header", "true")
        .csv(coupon_clip_path)
    )
df.createOrReplaceTempView('df')

## Save to delta table

In [0]:
spark.sql(f"""
    INSERT OVERWRITE {bronze_coupon_clip}
    SELECT
        usercode,
        EVENTDATETIME,
        EVENTTYPE,
        OFFERID,
        STORENAME,
        OFFERCODE,
        OFFERACTIVEDATE,
        OFFERSHUTOFFDATE,
        OFFEREXPIRYDATE,
        BRAND,
        OFFERVALUE,
        OFFERTYPE,
        DISCOUNTTYPE,
        CATEGORY,
        SOURCE,
        APPID,
        APPCODE
    FROM df
""")